做单独点的MC/MMCA运行图 beta=0.08/0.10

In [ ]:
#!/usr/bin/env python3
# hyper_mc_each_graph_one_run.py
"""
MC KAPPA=12 做上图一个点的所有时间步演化图
方案 B：每次生成一张新的 3-uniform 超图，并在该超图上做一次 Monte-Carlo（共 R 次）。
最终保存 rho_runs (R x (T+1)) 与 mean_rho_mc 到 CSV。
运行：python hyper_mc_each_graph_one_run.py
"""

import numpy as np
import random
import math
import pandas as pd
import matplotlib.pyplot as plt
from time import time

# -------------------- 超图生成器（与你原始实现一致） --------------------
def generate_hypergraph_H_n_m_3(n, kappa, seed=None, batch_size=3000):
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)
    d = 3
    m = int(round(n * kappa / d))
    max_possible = math.comb(n, d)
    if m > max_possible:
        raise ValueError("m > C(n,3)")
    nodes = np.arange(n)
    edges_set = set()
    while len(edges_set) < m:
        tris = np.random.choice(nodes, size=(batch_size, d), replace=True)
        for i in range(tris.shape[0]):
            row = tris[i]
            if len(np.unique(row)) < d:
                row = np.random.choice(nodes, size=(d,), replace=False)
            tris[i] = np.sort(row)
        for row in tris:
            if len(edges_set) >= m:
                break
            edges_set.add(tuple(row.tolist()))
    hyperedges = np.array(list(edges_set), dtype=int)
    hyperedges.sort(axis=1)
    return hyperedges

# -------------------- HyperSISModel（与你的实现一致） --------------------
class HyperSISModel:
    def __init__(self, hyperedges, n):
        self.hyperedges = np.asarray(hyperedges, dtype=int)
        self.n = n
        self.m = self.hyperedges.shape[0]
        self.edge_i = self.hyperedges[:,0]
        self.edge_j = self.hyperedges[:,1]
        self.edge_k = self.hyperedges[:,2]

        self.node_edge_pos = [[] for _ in range(self.n)]
        for ei in range(self.m):
            a,b,c = self.hyperedges[ei]
            self.node_edge_pos[a].append((ei,0))
            self.node_edge_pos[b].append((ei,1))
            self.node_edge_pos[c].append((ei,2))

    def _compute_edge_products(self, p):
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        return (p_j * p_k, p_i * p_k, p_i * p_j)

    def compute_qi(self, p, Theta, beta_d):
        prod_pos0, prod_pos1, prod_pos2 = self._compute_edge_products(p)
        qi = np.zeros(self.n, dtype=float)
        for node in range(self.n):
            prod_val = 1.0
            for (ei, pos) in self.node_edge_pos[node]:
                if pos == 0:
                    prod_oth = prod_pos0[ei]
                elif pos == 1:
                    prod_oth = prod_pos1[ei]
                else:
                    prod_oth = prod_pos2[ei]
                prod_val *= (1.0 - Theta[ei] * beta_d * prod_oth)
            qi[node] = 1.0 - prod_val
        return np.clip(qi, 0.0, 1.0)

    def _update_theta_from_p(self, p, Theta, eta, gamma):
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        Phi = p_i * p_j * p_k
        Theta_new = Theta * (1.0 - eta * Phi) + gamma * (1.0 - Theta)
        return np.clip(Theta_new, 0.0, 1.0)

    def run_mc_single(self, beta_d=0.3, mu=0.2, eta=1.0, gamma=0.05,
                      T=100, init_frac=0.1, seed=None, p_smooth=0.8, verbose=False):
        if seed is not None:
            np.random.seed(seed); random.seed(seed)
        states = np.zeros(self.n, dtype=int)
        initial_infected = max(1, int(round(init_frac * self.n)))
        init_nodes = np.random.choice(self.n, size=initial_infected, replace=False)
        states[init_nodes] = 1
        p = np.full(self.n, init_frac, dtype=float)
        Theta = np.ones(self.m, dtype=float)
        rho_ts = []
        theta_ts = []
        for t in range(T):
            rho_ts.append(states.mean()); theta_ts.append(Theta.mean())
            if p_smooth == 0.0:
                p = states.astype(float)
            else:
                alpha = float(p_smooth)
                p = alpha * p + (1.0 - alpha) * states
            qi = self.compute_qi(p, Theta, beta_d)
            susceptible_idx = np.where(states == 0)[0]
            rand_vals = np.random.rand(susceptible_idx.size)
            infected_hits = susceptible_idx[rand_vals < qi[susceptible_idx]]
            new_infections = set(int(x) for x in infected_hits)
            infected_idx = np.where(states == 1)[0]
            if infected_idx.size > 0:
                recovered_mask = np.random.rand(infected_idx.size) < mu
                recovered_nodes = infected_idx[recovered_mask]
            else:
                recovered_nodes = np.array([], dtype=int)
            next_states = states.copy()
            if recovered_nodes.size > 0:
                next_states[recovered_nodes] = 0
            for node in new_infections:
                if next_states[node] == 0:
                    next_states[node] = 1
            states = next_states
            Theta = self._update_theta_from_p(p, Theta, eta, gamma)
            if verbose and (t % 20 == 0):
                print(f"[MC] t={t:3d}  rho={rho_ts[-1]:.4f}  ⟨Θ⟩={theta_ts[-1]:.4f}")
        rho_ts.append(states.mean()); theta_ts.append(Theta.mean())
        return np.array(rho_ts), np.array(theta_ts)

# -------------------- 运行参数（修改部分） --------------------
n = 1500
kappa = 12.0

beta_1 = 0.08 # 第一个感染率
beta_2 = 0.11 # 第二个感染率

mu = 0.1
T = 300
init_frac = 0.3  
seed = 12345        # 基础种子
eta = 0.6
gamma = 0.02
p_smooth = 0.0

R = 120  # 重复次数

# -------------------- 主循环：为两个初始比例分别运行 --------------------
print("Starting: each repeat generates a new hypergraph and runs two MC with different init_frac.")
# 为两个初始比例分别创建存储数组
rho_runs_1 = np.zeros((R, T+1), dtype=float)  # init_frac = 0.08
rho_runs_2 = np.zeros((R, T+1), dtype=float)  # init_frac = 0.10

t0 = time()
for r in range(R):
    gen_seed = None if seed is None else (seed + r)
    hyperedges = generate_hypergraph_H_n_m_3(n=n, kappa=kappa, seed=gen_seed, batch_size=3000)
    model = HyperSISModel(hyperedges, n)

    # 为两个初始比例使用相同的随机种子，确保其他随机因素一致
    run_seed = None if seed is None else (seed + 1000 + r)
    
    # 运行第一个初始比例 (0.08)
    rho_r_1, theta_r_1 = model.run_mc_single(beta_d=beta_1, mu=mu, eta=eta, gamma=gamma,
                                             T=T, init_frac=init_frac, seed=run_seed,
                                             p_smooth=p_smooth, verbose=False)
    rho_runs_1[r] = rho_r_1
    
    # 运行第二个初始比例 (0.10) - 使用新的随机种子避免相同随机序列
    run_seed_2 = None if seed is None else (seed + 2000 + r)
    rho_r_2, theta_r_2 = model.run_mc_single(beta_d=beta_2, mu=mu, eta=eta, gamma=gamma,
                                             T=T, init_frac=init_frac, seed=run_seed_2,
                                             p_smooth=p_smooth, verbose=False)
    rho_runs_2[r] = rho_r_2

    if (r+1) % 10 == 0 or (r+1) == R:
        elapsed = time() - t0
        print(f"  Completed {r+1}/{R} runs, elapsed {elapsed:.1f}s")

t_mc = time() - t0
print(f"All {R} independent-graph runs done in {t_mc:.2f}s")

# -------------------- 统计量：计算两个初始比例的均值 --------------------
mean_rho_mc_1 = rho_runs_1.mean(axis=0)  # init_frac = 0.08 的平均
mean_rho_mc_2 = rho_runs_2.mean(axis=0)  # init_frac = 0.10 的平均

# -------------------- 保存数据到 CSV（修改部分） --------------------
ts = np.arange(T+1)
# 创建包含三列的DataFrame
results_df = pd.DataFrame({
    'time': ts, 
    'mean_rho_mc_1': mean_rho_mc_1,  # init_frac = 0.08 的结果
    'mean_rho_mc_2': mean_rho_mc_2   # init_frac = 0.10 的结果
})

# 保存到CSV文件
results_df.to_csv("./out_mc/rho_mean_mc_comparison.csv", index=False, float_format="%.4f")

print("Saved rho_mean_mc_comparison.csv with both init_frac results")

print(f"Shape of rho_runs_1: {rho_runs_1.shape}")
print(f"Shape of rho_runs_2: {rho_runs_2.shape}")

Starting: each repeat generates a new hypergraph and runs two MC with different init_frac.
  Completed 10/120 runs, elapsed 31.7s
  Completed 20/120 runs, elapsed 62.3s
  Completed 30/120 runs, elapsed 92.6s
  Completed 40/120 runs, elapsed 123.1s
  Completed 50/120 runs, elapsed 153.5s
  Completed 60/120 runs, elapsed 185.6s
  Completed 70/120 runs, elapsed 216.0s
  Completed 80/120 runs, elapsed 246.5s
  Completed 90/120 runs, elapsed 277.2s
  Completed 100/120 runs, elapsed 307.7s
  Completed 110/120 runs, elapsed 339.1s
  Completed 120/120 runs, elapsed 369.7s
All 120 independent-graph runs done in 369.71s
Saved rho_mean_mc_comparison.csv with both init_frac results
Shape of rho_runs_1: (120, 301)
Shape of rho_runs_2: (120, 301)


In [ ]:
#!/usr/bin/env python3
# hyper_mmca_each_graph.py
"""
MMCA KAPPA=12 做上图一个点的所有时间步演化图
方案 B：每次生成一张新的 3-uniform 超图并在其上运行一次 MMCA（共 R 次）。
同时运行两个初始感染比例 (0.08 和 0.10)，结果保存在同一个CSV中。
运行：python hyper_mmca_each_graph.py
"""

import numpy as np
import random
import math
import matplotlib.pyplot as plt
import pandas as pd
from time import time

def generate_hypergraph_H_n_m_3(n, kappa, seed=None, batch_size=3000):
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)
    d = 3
    m = int(round(n * kappa / d))
    max_possible = math.comb(n, d)
    if m > max_possible:
        raise ValueError("m > C(n,3)")
    nodes = np.arange(n)
    edges_set = set()
    while len(edges_set) < m:
        tris = np.random.choice(nodes, size=(batch_size, d), replace=True)
        for i in range(tris.shape[0]):
            row = tris[i]
            if len(np.unique(row)) < d:
                row = np.random.choice(nodes, size=(d,), replace=False)
            tris[i] = np.sort(row)
        for row in tris:
            if len(edges_set) >= m:
                break
            edges_set.add(tuple(row.tolist()))
    hyperedges = np.array(list(edges_set), dtype=int)
    hyperedges.sort(axis=1)
    return hyperedges

class HyperSISModel:
    def __init__(self, hyperedges, n):
        self.hyperedges = np.asarray(hyperedges, dtype=int)
        self.n = n
        self.m = self.hyperedges.shape[0]
        self.edge_i = self.hyperedges[:,0]
        self.edge_j = self.hyperedges[:,1]
        self.edge_k = self.hyperedges[:,2]

        self.node_edge_pos = [[] for _ in range(self.n)]
        for ei in range(self.m):
            a,b,c = self.hyperedges[ei]
            self.node_edge_pos[a].append((ei,0))
            self.node_edge_pos[b].append((ei,1))
            self.node_edge_pos[c].append((ei,2))

    def _compute_edge_products(self, p):
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        return (p_j * p_k, p_i * p_k, p_i * p_j)

    def compute_qi(self, p, Theta, beta_d):
        prod_pos0, prod_pos1, prod_pos2 = self._compute_edge_products(p)
        qi = np.zeros(self.n, dtype=float)
        for node in range(self.n):
            prod_val = 1.0
            for (ei, pos) in self.node_edge_pos[node]:
                if pos == 0:
                    prod_oth = prod_pos0[ei]
                elif pos == 1:
                    prod_oth = prod_pos1[ei]
                else:
                    prod_oth = prod_pos2[ei]
                prod_val *= (1.0 - Theta[ei] * beta_d * prod_oth)
            qi[node] = 1.0 - prod_val
        return np.clip(qi, 0.0, 1.0)

    def _update_theta_from_p(self, p, Theta, eta, gamma):
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        Phi = p_i * p_j * p_k
        Theta_new = Theta * (1.0 - eta * Phi) + gamma * (1.0 - Theta)
        return np.clip(Theta_new, 0.0, 1.0)

    def run_mmca(self, beta_d=0.3, mu=0.2, eta=1.0, gamma=0.05,
                 T=100, init_frac=0.01, verbose=False):
        p = np.full(self.n, init_frac, dtype=float)
        Theta = np.ones(self.m, dtype=float)
        rho_ts = []
        theta_ts = []
        for t in range(T):
            rho_ts.append(p.mean())
            theta_ts.append(Theta.mean())
            prod_pos0, prod_pos1, prod_pos2 = self._compute_edge_products(p)
            Q = np.ones(self.n, dtype=float)
            for node in range(self.n):
                prod_val = 1.0
                for (ei,pos) in self.node_edge_pos[node]:
                    if pos == 0:
                        prod_oth = prod_pos0[ei]
                    elif pos == 1:
                        prod_oth = prod_pos1[ei]
                    else:
                        prod_oth = prod_pos2[ei]
                    prod_val *= (1.0 - Theta[ei] * beta_d * prod_oth)
                Q[node] = prod_val
            p_next = (1.0 - p) * (1.0 - Q) + (1.0 - mu) * p
            Theta = self._update_theta_from_p(p, Theta, eta, gamma)
            p = p_next
            if verbose and (t % 20 == 0):
                print(f"[MMCA] t={t:3d}  rho={rho_ts[-1]:.4f}  ⟨Θ⟩={theta_ts[-1]:.4f}")
        rho_ts.append(p.mean()); theta_ts.append(Theta.mean())
        return np.array(rho_ts), np.array(theta_ts)

# -------------------- 参数（按您的要求修改） --------------------
n = 1500
kappa = 12.0        # 使用与MC相同的kappa
beta_1 = 0.08          # 使用与MC相同的beta
beta_2 = 0.12

mu = 0.1
T = 300             # 使用与MC相同的T
# 定义两个不同的初始感染比例
init_frac = 0.3
seed = 12345        # 基础种子
eta = 0.6           # 使用与MC相同的eta
gamma = 0.02        # 使用与MC相同的gamma

R = 80  # 重复次数：每次生成新网络并运行一次 MMCA

# -------------------- 主循环：每次生成新超图并在其上运行两次 MMCA（不同初始比例） --------------------
print("Starting: each repeat generates a new hypergraph and runs two MMCA with different init_frac.")
# 为两个初始比例分别创建存储数组
rho_runs_1 = np.zeros((R, T+1), dtype=float)  # init_frac = 0.08
rho_runs_2 = np.zeros((R, T+1), dtype=float)  # init_frac = 0.10

t0 = time()
for r in range(R):
    gen_seed = None if seed is None else (seed + r)   # 生成超图的种子（可复现）
    hyperedges = generate_hypergraph_H_n_m_3(n=n, kappa=kappa, seed=gen_seed, batch_size=3000)
    model = HyperSISModel(hyperedges, n)

    # 运行第一个初始比例 (0.08)
    rho_r_1, theta_r_1 = model.run_mmca(beta_d=beta_1, mu=mu, eta=eta, gamma=gamma,
                                        T=T, init_frac=init_frac, verbose=False)
    rho_runs_1[r] = rho_r_1

    # 运行第二个初始比例 (0.10) - 使用相同的超图
    rho_r_2, theta_r_2 = model.run_mmca(beta_d=beta_2, mu=mu, eta=eta, gamma=gamma,
                                        T=T, init_frac=init_frac, verbose=False)
    rho_runs_2[r] = rho_r_2


    if (r+1) % 10 == 0 or (r+1) == R:
        elapsed = time() - t0
        print(f"  Completed {r+1}/{R} runs, elapsed {elapsed:.1f}s")

t_mmca = time() - t0
print(f"All {R} independent-graph MMCA runs done in {t_mmca:.2f}s")

# -------------------- 统计量：计算两个初始比例的均值 --------------------
mean_rho_mmca_1 = rho_runs_1.mean(axis=0)  # init_frac = 0.08 的平均
mean_rho_mmca_2 = rho_runs_2.mean(axis=0)  # init_frac = 0.10 的平均


# -------------------- 保存数据到 CSV --------------------
ts = np.arange(T+1)

# 主结果文件：包含两个初始比例的rho均值
results_df = pd.DataFrame({
    'time': ts,
    'mean_rho_mmca_1': mean_rho_mmca_1,  # init_frac = 0.08 的结果
    'mean_rho_mmca_2': mean_rho_mmca_2   # init_frac = 0.10 的结果
})
results_df.to_csv("./out_mmca/rho_mean_mmca_comparison.csv", index=False, float_format="%.6f")


print("Saved comparison results to ./out_mmca/")
print(f"Shape of rho_runs_1: {rho_runs_1.shape}")
print(f"Shape of rho_runs_2: {rho_runs_2.shape}")

Starting: each repeat generates a new hypergraph and runs two MMCA with different init_frac.
  Completed 10/80 runs, elapsed 29.3s
  Completed 20/80 runs, elapsed 58.4s
  Completed 30/80 runs, elapsed 87.5s
  Completed 40/80 runs, elapsed 117.1s
  Completed 50/80 runs, elapsed 146.4s
  Completed 60/80 runs, elapsed 175.9s
  Completed 70/80 runs, elapsed 205.3s
  Completed 80/80 runs, elapsed 234.7s
All 80 independent-graph MMCA runs done in 234.70s
Saved comparison results to ./out_mmca/
Shape of rho_runs_1: (80, 301)
Shape of rho_runs_2: (80, 301)
